In [ ]:
import pandas as pd
import re
import os
import pingouin as pg

def get_parsed_sfcn(filepath):
    data = []
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return pd.DataFrame()
    
    with open(filepath, 'r') as f:
        for line in f:
            # Captures first 4 digits as subject_id, next 2 as age_digit
            match = re.search(r"sub-(\d{4})(\d{2}): Actual: ([\d.]+), Predicted: ([\d.]+)", line)
            if match:
                data.append({
                    "subject_id": f"sub-{match.group(1)}", 
                    "SFCN": float(match.group(4)),
                    "age": float(match.group(3))
                })
    return pd.DataFrame(data)

def run_sfcn_stability(df, label):
    df['SFCN_BAG'] = df['SFCN'] - df['age']
    
    counts = df['subject_id'].value_counts()
    long_subs = counts[counts > 1].index
    df_long = df[df['subject_id'].isin(long_subs)].copy()
    
    if df_long.empty:
        return f"No longitudinal data found for {label}."

    df_long['session'] = df_long.groupby('subject_id')['age'].rank(method='first')

    icc = pg.intraclass_corr(data=df_long, targets='subject_id', 
                            raters='session', ratings='SFCN_BAG').set_index('Type')
    
    return {
        'Cohort': label,
        'ICC': icc.loc['ICC3', 'ICC'].round(3),
        '95% CI': icc.loc['ICC3', 'CI95%'],
        'Subjects_N': len(long_subs)
    }


df_peds_sfcn = get_parsed_sfcn('logs_real/Pediatric/SFCN_Pediatric_Real.txt')
df_adult_sfcn = get_parsed_sfcn('logs_real/Adult/SFCN_Adult_Real.txt')

peds_results = run_sfcn_stability(df_peds_sfcn, "Pediatric")
adult_results = run_sfcn_stability(df_adult_sfcn, "Adult")

print("--- SFCN BAG Stability (ICC) ---")
print(pd.DataFrame([peds_results, adult_results]))

--- SFCN BAG Stability (ICC) ---
      Cohort    ICC        95% CI  Subjects_N
0  Pediatric  0.673  [0.63, 0.71]         250
1      Adult  0.882   [0.86, 0.9]         250


In [ ]:
import pandas as pd
import os
import pingouin as pg

def load_and_split_vox1(filepath):
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return pd.DataFrame()
    
    df = pd.read_csv(filepath)
    df = df.rename(columns={'subj_id': 'participant_id', 'predicted_age_mean': 'VOX1'})
    
    df['subject_id'] = df['participant_id'].str.extract(r'sub-(\d{4})')[0].apply(lambda x: f"sub-{x}")
    
    return df[['subject_id', 'VOX1', 'age']]

def run_vox1_stability(df, label):
    df['VOX1_BAG'] = df['VOX1'] - df['age']
    
    counts = df['subject_id'].value_counts()
    long_subs = counts[counts > 1].index
    df_long = df[df['subject_id'].isin(long_subs)].copy()
    
    if df_long.empty:
        return f"No longitudinal data found for {label} (VOX1)."

    df_long['session'] = df_long.groupby('subject_id')['age'].rank(method='first')

    icc = pg.intraclass_corr(data=df_long, targets='subject_id', 
                            raters='session', ratings='VOX1_BAG').set_index('Type')
    
    return {
        'Cohort': label,
        'Model': 'VOX1',
        'ICC_BAG': icc.loc['ICC3', 'ICC'].round(3),
        '95% CI': icc.loc['ICC3', 'CI95%'],
        'Subjects_N': len(long_subs)
    }


df_peds_v1 = load_and_split_vox1('vox1_age_results_pediatric.csv')
df_adult_v1 = load_and_split_vox1('vox1_age_results_adult.csv')

v1_peds_res = run_vox1_stability(df_peds_v1, "Pediatric")
v1_adult_res = run_vox1_stability(df_adult_v1, "Adult")

print("--- VOX1 BAG Stability (ICC) ---")
results_df = pd.DataFrame([v1_peds_res, v1_adult_res])
print(results_df)

--- VOX1 BAG Stability (ICC) ---
      Cohort Model  ICC_BAG        95% CI  Subjects_N
0  Pediatric  VOX1    0.647   [0.6, 0.69]         250
1      Adult  VOX1    0.855  [0.83, 0.88]         250


In [ ]:
import pandas as pd
import os
import pingouin as pg

def load_and_split_vox2(filepath):
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return pd.DataFrame()
    
    df = pd.read_csv(filepath)
    df = df.rename(columns={'subj_id': 'participant_id', 'predicted_age_mean': 'VOX2'})
    

    df['subject_id'] = df['participant_id'].str.extract(r'sub-(\d{4})')[0].apply(lambda x: f"sub-{x}")
    
    return df[['subject_id', 'VOX2', 'age']]

def run_vox2_stability(df, label):
    df['VOX2_BAG'] = df['VOX2'] - df['age']
    
    counts = df['subject_id'].value_counts()
    long_subs = counts[counts > 1].index
    df_long = df[df['subject_id'].isin(long_subs)].copy()
    
    if df_long.empty:
        return f"No longitudinal data found for {label} (VOX2)."

    df_long['session'] = df_long.groupby('subject_id')['age'].rank(method='first')

    icc = pg.intraclass_corr(data=df_long, targets='subject_id', 
                            raters='session', ratings='VOX2_BAG').set_index('Type')
    
    return {
        'Cohort': label,
        'Model': 'VOX2',
        'ICC_BAG': icc.loc['ICC3', 'ICC'].round(3),
        '95% CI': icc.loc['ICC3', 'CI95%'],
        'Subjects_N': len(long_subs)
    }

df_peds_v2 = load_and_split_vox2('vox2_age_results_pediatric.csv')
df_adult_v2 = load_and_split_vox2('vox2_age_results_adult.csv')

v2_peds_res = run_vox2_stability(df_peds_v2, "Pediatric")
v2_adult_res = run_vox2_stability(df_adult_v2, "Adult")

print("--- VOX2 BAG Stability (ICC) ---")
print(pd.DataFrame([v2_peds_res, v2_adult_res]))

--- VOX2 BAG Stability (ICC) ---
      Cohort Model  ICC_BAG        95% CI  Subjects_N
0  Pediatric  VOX2    0.582  [0.54, 0.63]         250
1      Adult  VOX2    0.944  [0.93, 0.95]         250
